# Notebook 07 - Combined Feature Model

Goal:

Test whether combining character n-gram features with handcrafted lexical features improves domain classification.

Previous best useful model from Notebook 06:

Character N-Gram + Logistic Regression

- Accuracy: ~0.881
- Macro F1: ~0.38
- Malware recall: ~0.54
- Phishing recall: ~0.48
- Spam recall: ~0.34

This notebook tests:

domain text -> character n-grams  
numeric lexical features -> scaled  
TLD -> one-hot encoded  
classifier -> Logistic Regression / SGD

Primary goal:

Improve minority-class detection, especially malware, phishing, and spam.

In [2]:
import pandas as pd
import numpy as np
import joblib
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression, SGDClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    precision_score
)

In [3]:
DATA_PATH = "../data/processed/auspex_features_v1.csv"

features = pd.read_csv(DATA_PATH)

print(features.shape)
features.head()

(1024538, 11)


,domain,label,length,digit_count,digit_ratio,dot_count,tld,starts_with_digit,contains_www,entropy,hyphen_count
0,cypress.com,benign,11,0,0.000000,1,com,0,0,3.095795,0
1,boy.jp,benign,6,0,0.000000,1,jp,0,0,2.584963,0
2,nnm.ru,benign,6,0,0.000000,1,ru,0,0,2.251629,0
3,1stwebdesigner.com,benign,18,1,0.055556,1,com,1,0,3.794653,0
4,ordasoft.com,benign,12,0,0.000000,1,com,0,0,3.188722,0


In [4]:
expected_columns = [
    "domain",
    "label",
    "length",
    "digit_count",
    "digit_ratio",
    "dot_count",
    "tld",
    "starts_with_digit",
    "contains_www",
    "entropy",
    "hyphen_count",
]

missing_columns = set(expected_columns) - set(features.columns)
extra_columns = set(features.columns) - set(expected_columns)

print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

print("\nNull counts:")
print(features[expected_columns].isna().sum())

print("\nClass counts:")
print(features["label"].value_counts())

print("\nDuplicate domains:")
print(features["domain"].duplicated().sum())

assert len(missing_columns) == 0, "Missing expected columns."
assert "label" in features.columns, "Missing label column."
assert "domain" in features.columns, "Missing domain column."

Missing columns: set()
Extra columns: set()

Null counts:
domain               0
label                0
length               0
digit_count          0
digit_ratio          0
dot_count            0
tld                  0
starts_with_digit    0
contains_www         0
entropy              0
hyphen_count         0
dtype: int64

Class counts:
label
benign      988299
malware      26703
phishing      8705
spam           831
Name: count, dtype: int64

Duplicate domains:
0


In [5]:
features = features.copy()

features["domain"] = features["domain"].astype(str).str.lower().str.strip()
features["label"] = features["label"].astype(str).str.lower().str.strip()
features["tld"] = features["tld"].fillna("unknown").astype(str).str.lower().str.strip()

numeric_features = [
    "length",
    "digit_count",
    "digit_ratio",
    "dot_count",
    "starts_with_digit",
    "contains_www",
    "entropy",
    "hyphen_count",
]

for col in numeric_features:
    features[col] = pd.to_numeric(features[col], errors="coerce")

features[numeric_features] = features[numeric_features].fillna(0)

print(features.dtypes)
print("\nRemaining nulls:")
print(features.isna().sum())

domain                   str
label                    str
length                 int64
digit_count            int64
digit_ratio          float64
dot_count              int64
tld                      str
starts_with_digit      int64
contains_www           int64
entropy              float64
hyphen_count           int64
dtype: object

Remaining nulls:
domain               0
label                0
length               0
digit_count          0
digit_ratio          0
dot_count            0
tld                  0
starts_with_digit    0
contains_www         0
entropy              0
hyphen_count         0
dtype: int64


In [6]:
X = features.drop(columns=["label"])
y = features["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

print("\nTrain distribution:")
print(y_train.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

Train size: 819630
Test size: 204908

Train distribution:
label
benign      790639
malware      21362
phishing      6964
spam           665
Name: count, dtype: int64

Test distribution:
label
benign      197660
malware       5341
phishing      1741
spam           166
Name: count, dtype: int64


In [7]:
LABEL_ORDER = ["benign", "malware", "phishing", "spam"]

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)

    print(model_name)
    print("-" * len(model_name))

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Weighted F1:", weighted_f1)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=LABEL_ORDER,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred, labels=LABEL_ORDER)

    print("\nConfusion Matrix:")
    print(cm)

    threat_true = y_test != "benign"
    threat_pred = y_pred != "benign"

    malicious_recall = recall_score(
        threat_true,
        threat_pred,
        pos_label=True,
        zero_division=0
    )

    malicious_precision = precision_score(
        threat_true,
        threat_pred,
        pos_label=True,
        zero_division=0
    )

    malicious_f1 = f1_score(
        threat_true,
        threat_pred,
        pos_label=True,
        zero_division=0
    )

    threats_predicted_benign = ((y_test != "benign") & (y_pred == "benign")).sum()
    total_threats = (y_test != "benign").sum()

    print("\nBinary Threat Detection View:")
    print("Malicious recall:", malicious_recall)
    print("Malicious precision:", malicious_precision)
    print("Malicious F1:", malicious_f1)
    print("Threats predicted benign:", threats_predicted_benign, "/", total_threats)

    return {
        "model_name": model_name,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "malicious_recall": malicious_recall,
        "malicious_precision": malicious_precision,
        "malicious_f1": malicious_f1,
        "threats_predicted_benign": int(threats_predicted_benign),
        "total_threats": int(total_threats),
        "confusion_matrix": cm.tolist()
    }

In [8]:
notebook_06_benchmark = {
    "model_name": "Notebook 06 - Pure N-Gram Logistic Regression",
    "accuracy": 0.8810734573564721,
    "macro_f1": 0.38,
    "malware_recall": 0.54,
    "phishing_recall": 0.48,
    "spam_recall": 0.34,
}

notebook_06_benchmark

{'model_name': 'Notebook 06 - Pure N-Gram Logistic Regression',
 'accuracy': 0.8810734573564721,
 'macro_f1': 0.38,
 'malware_recall': 0.54,
 'phishing_recall': 0.48,
 'spam_recall': 0.34}

In [9]:
combined_preprocessor = ColumnTransformer(
    transformers=[
        (
            "domain_ngrams",
            HashingVectorizer(
                analyzer="char",
                ngram_range=(2, 5),
                n_features=2**18,
                alternate_sign=False
            ),
            "domain"
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "tld",
            OneHotEncoder(handle_unknown="ignore"),
            ["tld"]
        ),
    ],
    remainder="drop"
)

In [10]:
sample_X = X_train.head(1000)

smoke_preprocessor = clone(combined_preprocessor)
sample_transformed = smoke_preprocessor.fit_transform(sample_X)

print("Sample transformed shape:", sample_transformed.shape)
print("Sample transformed type:", type(sample_transformed))

if hasattr(sample_transformed, "nnz"):
    density = sample_transformed.nnz / (sample_transformed.shape[0] * sample_transformed.shape[1])
    print("Sample sparse density:", density)

Sample transformed shape: (1000, 262250)
Sample transformed type: <class 'scipy.sparse._csr.csr_matrix'>
Sample sparse density: 0.00021482173498570066


In [11]:
combined_sgd = Pipeline([
    ("preprocessor", combined_preprocessor),
    ("classifier", SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        max_iter=30,
        random_state=42,
        n_jobs=-1
    ))
])

combined_sgd.fit(X_train, y_train)

print("Combined SGD training complete.")

Combined SGD training complete.


c:\Users\pipza\OneDrive\Desktop\Project-Auspex\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


In [12]:
combined_sgd_results = evaluate_model(
    combined_sgd,
    X_test,
    y_test,
    "Combined Character N-Grams + Lexical Features + SGD"
)

combined_sgd_results

Combined Character N-Grams + Lexical Features + SGD
---------------------------------------------------
Accuracy: 0.966194584886876
Macro F1: 0.4538081624418597
Weighted F1: 0.9621005235844521

Classification Report:
              precision    recall  f1-score   support

      benign       0.98      0.99      0.98    197660
     malware       0.62      0.22      0.32      5341
    phishing       0.47      0.42      0.44      1741
        spam       0.04      0.21      0.06       166

    accuracy                           0.97    204908
   macro avg       0.53      0.46      0.45    204908
weighted avg       0.96      0.97      0.96    204908


Confusion Matrix:
[[196047    560    279    774]
 [  3518   1164    550    109]
 [   829    163    735     14]
 [   129      2      0     35]]

Binary Threat Detection View:
Malicious recall: 0.3824503311258278
Malicious precision: 0.6321550741163056
Malicious F1: 0.4765752600361042
Threats predicted benign: 4476 / 7248


{'model_name': 'Combined Character N-Grams + Lexical Features + SGD',
 'accuracy': 0.966194584886876,
 'macro_f1': 0.4538081624418597,
 'weighted_f1': 0.9621005235844521,
 'malicious_recall': 0.3824503311258278,
 'malicious_precision': 0.6321550741163056,
 'malicious_f1': 0.4765752600361042,
 'threats_predicted_benign': 4476,
 'total_threats': 7248,
 'confusion_matrix': [[196047, 560, 279, 774],
  [3518, 1164, 550, 109],
  [829, 163, 735, 14],
  [129, 2, 0, 35]]}

In [13]:
combined_logreg = Pipeline([
    ("preprocessor", combined_preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        solver="saga",
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ))
])

combined_logreg.fit(X_train, y_train)

print("Combined Logistic Regression training complete.")

c:\Users\pipza\OneDrive\Desktop\Project-Auspex\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Combined Logistic Regression training complete.


c:\Users\pipza\OneDrive\Desktop\Project-Auspex\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [14]:
combined_logreg_results = evaluate_model(
    combined_logreg,
    X_test,
    y_test,
    "Combined Character N-Grams + Lexical Features + Logistic Regression"
)

combined_logreg_results

Combined Character N-Grams + Lexical Features + Logistic Regression
-------------------------------------------------------------------
Accuracy: 0.8505670837644211
Macro F1: 0.35286912527456316
Weighted F1: 0.8944760845548433

Classification Report:
              precision    recall  f1-score   support

      benign       0.99      0.86      0.92    197660
     malware       0.13      0.58      0.22      5341
    phishing       0.12      0.51      0.20      1741
        spam       0.04      0.48      0.08       166

    accuracy                           0.85    204908
   macro avg       0.32      0.60      0.35    204908
weighted avg       0.96      0.85      0.89    204908


Confusion Matrix:
[[170251  19793   5979   1637]
 [  1800   3076    413     52]
 [   487    360    882     12]
 [    44     38      5     79]]

Binary Threat Detection View:
Malicious recall: 0.6783940397350994
Malicious precision: 0.1521066633669492
Malicious F1: 0.248496487592864
Threats predicted benign: 2331

{'model_name': 'Combined Character N-Grams + Lexical Features + Logistic Regression',
 'accuracy': 0.8505670837644211,
 'macro_f1': 0.35286912527456316,
 'weighted_f1': 0.8944760845548433,
 'malicious_recall': 0.6783940397350994,
 'malicious_precision': 0.1521066633669492,
 'malicious_f1': 0.248496487592864,
 'threats_predicted_benign': 2331,
 'total_threats': 7248,
 'confusion_matrix': [[170251, 19793, 5979, 1637],
  [1800, 3076, 413, 52],
  [487, 360, 882, 12],
  [44, 38, 5, 79]]}